# Milestone 5: Scaling with PySpark & Interactive AML Intelligence Dashboard

## 1. Executive Motivation: Scaling Beyond Single-Machine Limits

In **Milestones 1 through 4**, we developed a comprehensive anti-money laundering detection pipeline on the **HI-Small** dataset (~5.08M transactions, 515,080 accounts):
- Hand-crafted graph features + XGBoost baseline (**PR-AUC: 0.0834 - 0.0914**)
- Inductive 2-layer GraphSAGE GNN with neighbor sampling (**PR-AUC: 0.1884**, +128% relative lift, 2.4x alert precision)
- Algorithmic temporal motif search for multi-hop layering chains ($A \to B \to C \to D$)
- Model interpretability via `GNNExplainer` and SAR-ready Analyst Risk Cards.

### The Challenge of Real-World Scale:
At major global financial institutions, transaction volumes reach **billions of edges** and tens of millions of accounts. Single-machine Pandas operations hit out-of-memory (OOM) bottlenecks:
1. **Distributed Feature Extraction:** Requires distributed DataFrames (PySpark) with partitioned window aggregations.
2. **Distributed Motif Matching:** Requires GraphFrames declarative pattern matching (`.find()`) across worker clusters.
3. **Operational UI:** A compliance intelligence dashboard (Streamlit) enabling sub-second risk lookups and SAR narrative generation.

## 2. Re-implementing Feature Engineering in PySpark

Below we demonstrate the distributed PySpark feature extraction pipeline ([`src/spark_features.py`](../src/spark_features.py)).

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

from src.spark_features import create_spark_session, compute_spark_account_features

# Initialize local PySpark session
spark = create_spark_session()
csv_path = '../data/processed/processed_df.csv'

# Extract features on sample of transactions using distributed window aggregations
spark_features = compute_spark_account_features(spark, csv_path, max_rows=50000)
print(f"Extracted distributed features for {spark_features.count():,} unique accounts.")
spark_features.select('account', 'in_degree', 'out_degree', 'total_in', 'total_out', 'avg_holding_time', 'unique_counterparties', 'is_laundering').show(5)

## 3. Distributed GraphFrames Motif Matching Architecture

In PySpark with GraphFrames, multi-hop laundering chains can be queried declaratively using GraphFrames' `.find()` pattern matcher:

```python
# Example 3-Hop Layering Chain Query in GraphFrames
patterns = graph.find("(a)-[e1]->(b); (b)-[e2]->(c); (c)-[e3]->(d)") \
    .filter("e2.Timestamp > e1.Timestamp AND e3.Timestamp > e2.Timestamp") \
    .filter("e2.amount <= e1.amount AND e2.amount >= 0.8 * e1.amount") \
    .filter("e3.amount <= e2.amount AND e3.amount >= 0.8 * e2.amount")
```

This allows parallel graph pattern matching to scale seamlessly across thousands of compute nodes on AWS EMR or Databricks.

## 4. Interactive AML Compliance Intelligence Dashboard (`app.py`)

To make the trained GraphSAGE model, chain detector, and GNNExplainer usable by non-technical compliance investigators, we packaged the entire system into an interactive Streamlit application ([`app.py`](../app.py)).

### Features of the Live Dashboard:
1. **Global Telemetry:** Monitored accounts, transactions, PR-AUC lifts, and alert precision metrics.
2. **Case Study Presets & Account Search:** Quick evaluation of known laundering intermediaries, high-velocity hubs, and clean commercial accounts.
3. **SAR-Ready Compliance Cards:** Downloadable plain-English narratives tailored for immediate regulatory filing.
4. **Multi-Hop Chain Stepper:** Visualizing the sequential flow of money, fee haircuts, and retention percentages.
5. **GNNExplainer Attribution Visualizer:** Dynamic bar charts of behavioral drivers and ranked counterparty flows.
6. **2-Hop Risk Ego-Networks:** High-resolution topological network visualization.

### Launching the Dashboard Locally:
```bash
streamlit run app.py
```

## 5. End-to-End System Benchmark & Project Retrospective

### Complete Benchmark Summary Across Milestones:

| Milestone | Model / Technique | Val PR-AUC | Test PR-AUC | Precision | Recall | Key Innovation |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Baseline Floor** | Random Guessing | 0.0123 | 0.0123 | 0.0123 | 1.0000 | 1.23% positive prevalence benchmark |
| **Milestone 2** | Tabular XGBoost (80% train) | — | 0.0914 | 0.0500 | 0.6700 | 8 hand-crafted graph features (7.4x lift) |
| **Milestone 3** | Retrained XGBoost (70% train) | 0.0855 | 0.0834 | 0.1178 | 0.1880 | Fair val-calibrated max-F1 threshold protocol |
| **Milestone 3** | **GraphSAGE GNN (2-layer)** | **0.2091** | **0.1884** | **0.2827** | **0.1495** | **+128% PR-AUC lift; 2.4x alert purity** |
| **Milestone 4** | **Temporal Chain Search + GNNExplainer** | N/A | N/A | N/A | N/A | Multi-hop motif matching + SAR Risk Cards |
| **Milestone 5** | **PySpark + Streamlit Dashboard** | N/A | N/A | N/A | N/A | Distributed feature pipeline + clickable UI |

### Honest Limitations & Practical Production Takeaways:
1. **Graph Depth Trade-offs:** GNNs excel at 2-hop local topology, but deep chains (3-5 hops) require algorithmic motif matching to prevent over-smoothing.
2. **Single-Institution Visibility:** In real-world banking, laundering chains often cross financial institutions; cross-bank federated learning or shared graph intelligence is required for end-to-end tracing.
3. **Human-in-the-Loop AI:** High alert purity (28.3%) drastically reduces compliance alert fatigue, but model outputs serve as an investigator prioritization assistant, not an automated decision-maker.